# EDA: パッチ・オーグメンテーション可視化

アノテーション点・パッチ抽出・augmentation の動作をインタラクティブに確認する。

**操作方法:**
- `idx` — manifest の行インデックス
- `aug_flags` — augmentation ビットフラグ (0=なし, 127=全て)
- `n_aug` — aug がある場合に表示するパッチ枚数
- `patch_t_min/max` / `perp_scale` / `patch_h/w` — パッチ切り出しパラメータ
- `label_t_min/max` — ラベル有効範囲（FLIP aug の折り返し点にも使用）
- **Random ボタン群** — フィルタ付きランダムジャンプ

In [ ]:
import sys
from pathlib import Path

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import math
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from matplotlib.gridspec import GridSpec

from eyeliner.data.dataset import load_manifest
from eyeliner.data.patch_utils import extract_patch
from eyeliner.data.augmentation import augment, augment_endpoints

print('ROOT:', ROOT)

In [ ]:
MANIFEST_PATH = ROOT / 'experiments/EXP017_rebalancing/manifests/manifest_train.csv'

records = load_manifest(MANIFEST_PATH)


def compute_meta(records: list[dict]) -> None:
    eye_points: dict = defaultdict(list)
    eye_count: dict = defaultdict(int)
    for rec in records:
        key = (rec['image_path'], rec['part_name'])
        eye_points[key].append((float(rec['px']), float(rec['py'])))
        eye_points[key].append((float(rec['qx']), float(rec['qy'])))
        eye_count[key] += 1

    eye_scale_map: dict = {}
    for key, pts in eye_points.items():
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        scale = math.sqrt((max(xs) - min(xs)) ** 2 + (max(ys) - min(ys)) ** 2)
        eye_scale_map[key] = max(scale, 1.0)

    for rec in records:
        key = (rec['image_path'], rec['part_name'])
        pq_dist = math.sqrt(
            (float(rec['qx']) - float(rec['px'])) ** 2
            + (float(rec['qy']) - float(rec['py'])) ** 2
        )
        eye_scale = eye_scale_map[key]
        n_pq = eye_count[key]
        rec['pq_dist'] = pq_dist
        rec['eye_scale'] = eye_scale
        rec['pq_ratio'] = pq_dist / eye_scale
        rec['weight'] = pq_dist / (eye_scale * n_pq)
        rec['val_weight'] = pq_dist / n_pq


compute_meta(records)

df = pd.DataFrame(records)
for col in ['t', 'px', 'py', 'qx', 'qy', 'pq_dist', 'eye_scale', 'pq_ratio']:
    df[col] = df[col].astype(float)
df['point_index'] = df['point_index'].astype(int)

n_uniq = df['image_path'].nunique()
print(f'サンプル数: {len(df):,}')
print(f'ユニーク画像数: {n_uniq:,}')

In [ ]:
_image_cache: dict = {}


def load_image(image_path: str):
    if image_path not in _image_cache:
        bgr = cv2.imread(str(ROOT / image_path))
        _image_cache[image_path] = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr is not None else None
    return _image_cache[image_path]


def patch_corners_in_image(
    px: float, py: float, qx: float, qy: float,
    patch_t_min: float, patch_t_max: float,
    patch_perp_scale: float, patch_h: int, patch_w: int,
):
    dx, dy = qx - px, qy - py
    norm = float(np.hypot(dx, dy))
    if norm == 0:
        return None
    ux, uy = dx / norm, dy / norm
    sx = patch_t_max * px + (1 - patch_t_max) * qx
    sy = patch_t_max * py + (1 - patch_t_max) * qy
    corners = []
    for pc_x, pc_y in [(0, 0), (patch_w - 1, 0), (patch_w - 1, patch_h - 1), (0, patch_h - 1)]:
        s_para = pc_x * (patch_t_max - patch_t_min) * norm / (patch_w - 1)
        d = (pc_y - patch_h // 2) * patch_perp_scale * norm / (patch_h - 1)
        corners.append((sx + s_para * ux + d * (-uy), sy + s_para * uy + d * ux))
    return corners


def t_to_patch_x(t_val: float, patch_t_min: float, patch_t_max: float, patch_w: int) -> float:
    return (patch_t_max - t_val) / (patch_t_max - patch_t_min) * (patch_w - 1)


def sample_with_aug(
    img, px, py, qx, qy, t_actual,
    patch_t_min, patch_t_max, patch_perp_scale,
    patch_h, patch_w, aug_flags, label_t_min, label_t_max,
):
    px_, py_, qx_, qy_, t_ = augment_endpoints(
        px, py, qx, qy, t_actual, label_t_min, label_t_max, aug_flags
    )
    patch = extract_patch(
        img, px_, py_, qx_, qy_,
        patch_size=(patch_h, patch_w),
        t_min=patch_t_min, t_max=patch_t_max,
        perp_scale=patch_perp_scale,
    )
    patch, t_aug = augment(patch, t_, t_min=label_t_min, t_max=label_t_max, aug_flags=aug_flags)
    return patch, t_aug

In [ ]:
w_idx = widgets.BoundedIntText(
    value=0, min=0, max=len(df) - 1,
    description='idx:', layout=widgets.Layout(width='200px'),
)
w_aug_flags = widgets.IntSlider(
    value=127, min=0, max=127, step=1,
    description='aug_flags:',
    style={'description_width': '80px'}, layout=widgets.Layout(width='350px'),
)
w_n_aug = widgets.IntSlider(
    value=6, min=1, max=9, step=1,
    description='n_aug:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='250px'),
)
w_t_min = widgets.FloatSlider(
    value=-2.0, min=-3.0, max=0.0, step=0.1,
    description='patch_t_min:', readout_format='.1f',
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'),
)
w_t_max = widgets.FloatSlider(
    value=3.0, min=1.0, max=5.0, step=0.1,
    description='patch_t_max:', readout_format='.1f',
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'),
)
w_perp = widgets.FloatSlider(
    value=2.0, min=0.1, max=5.0, step=0.1,
    description='perp_scale:', readout_format='.1f',
    style={'description_width': '80px'}, layout=widgets.Layout(width='350px'),
)
w_h = widgets.IntSlider(
    value=64, min=16, max=256, step=8,
    description='patch_h:',
    style={'description_width': '70px'}, layout=widgets.Layout(width='350px'),
)
w_w = widgets.IntSlider(
    value=64, min=16, max=256, step=8,
    description='patch_w:',
    style={'description_width': '70px'}, layout=widgets.Layout(width='350px'),
)
w_label_t_min = widgets.FloatSlider(
    value=-1.0, min=-3.0, max=0.0, step=0.1,
    description='label_t_min:', readout_format='.1f',
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'),
)
w_label_t_max = widgets.FloatSlider(
    value=2.0, min=1.0, max=5.0, step=0.1,
    description='label_t_max:', readout_format='.1f',
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'),
)

btn_rnd_all   = widgets.Button(description='Random (全体)',               button_style='info')
btn_rnd_upper = widgets.Button(description='Random (upper)',              button_style='')
btn_rnd_lower = widgets.Button(description='Random (lower)',              button_style='')
btn_rnd_edge  = widgets.Button(description='Random (端点 t<0.1 or >0.9)', button_style='warning')
btn_rnd_small = widgets.Button(description='Random (小eye_scale)',         button_style='danger')
btn_rnd_large = widgets.Button(description='Random (大eye_scale)',         button_style='success')

q20 = df['eye_scale'].quantile(0.20)
q80 = df['eye_scale'].quantile(0.80)


def pick_random(mask=None):
    subset = df[mask] if mask is not None else df
    if len(subset) == 0:
        return
    w_idx.value = int(subset.sample(1).index[0])


btn_rnd_all.on_click(lambda _: pick_random())
btn_rnd_upper.on_click(lambda _: pick_random(df['sub_key'] == 'upper'))
btn_rnd_lower.on_click(lambda _: pick_random(df['sub_key'] == 'lower'))
btn_rnd_edge.on_click(lambda _: pick_random((df['t'] < 0.1) | (df['t'] > 0.9)))
btn_rnd_small.on_click(lambda _: pick_random(df['eye_scale'] < q20))
btn_rnd_large.on_click(lambda _: pick_random(df['eye_scale'] > q80))

print(f'p20={q20:.1f}px  /  p80={q80:.1f}px')

In [ ]:
def render(
    idx, aug_flags, n_aug,
    patch_t_min, patch_t_max, patch_perp_scale,
    patch_h, patch_w, label_t_min, label_t_max,
):
    if idx >= len(df):
        print(f'idx {idx} はデータ範囲外')
        return

    rec        = df.iloc[idx]
    px, py     = float(rec['px']), float(rec['py'])
    qx, qy     = float(rec['qx']), float(rec['qy'])
    t_actual   = float(np.clip(rec['t'], label_t_min, label_t_max))
    eye_scale  = float(rec['eye_scale'])
    part_name  = str(rec['part_name'])
    sub_key    = str(rec['sub_key'])
    pq_ratio   = float(rec['pq_ratio'])
    image_path = str(rec['image_path'])

    img = load_image(image_path)
    if img is None:
        print(f'画像が見つかりません: {image_path}')
        return

    # 顔クロップ領域
    corners = patch_corners_in_image(
        px, py, qx, qy, patch_t_min, patch_t_max, patch_perp_scale, patch_h, patch_w
    )
    margin = max(80, int(eye_scale * 0.5))
    all_x = [px, qx] + ([c[0] for c in corners] if corners else [])
    all_y = [py, qy] + ([c[1] for c in corners] if corners else [])
    x1 = max(0, int(min(all_x)) - margin)
    y1 = max(0, int(min(all_y)) - margin)
    x2 = min(img.shape[1], int(max(all_x)) + margin)
    y2 = min(img.shape[0], int(max(all_y)) + margin)
    cropped = img[y1:y2, x1:x2]

    # パッチ生成
    if aug_flags == 0:
        patch_clean = extract_patch(
            img, px, py, qx, qy,
            patch_size=(patch_h, patch_w),
            t_min=patch_t_min, t_max=patch_t_max,
            perp_scale=patch_perp_scale,
        )
        patches_to_show = [(patch_clean, t_actual)]
        ncols_patch = 1
    else:
        patches_to_show = [
            sample_with_aug(
                img, px, py, qx, qy, t_actual,
                patch_t_min, patch_t_max, patch_perp_scale,
                patch_h, patch_w, aug_flags, label_t_min, label_t_max,
            )
            for _ in range(n_aug)
        ]
        ncols_patch = min(n_aug, 3)

    nrows_patch = math.ceil(len(patches_to_show) / ncols_patch)
    fig = plt.figure(figsize=(5 + 3 * ncols_patch, max(5, 3 * nrows_patch)))
    gs = GridSpec(nrows_patch, 1 + ncols_patch, figure=fig, hspace=0.35, wspace=0.3)

    # 左: 顔クロップ
    ax_face = fig.add_subplot(gs[:, 0])
    ax_face.imshow(cropped)
    if corners:
        poly_pts = np.array([(c[0] - x1, c[1] - y1) for c in corners])
        ax_face.add_patch(plt.Polygon(
            poly_pts, closed=True, fill=False,
            edgecolor='yellow', linewidth=2, linestyle='--',
        ))
    ax_face.plot(px - x1, py - y1, 'bo', markersize=9, label='P (t=1)', zorder=5)
    ax_face.plot(qx - x1, qy - y1, 'ro', markersize=9, label='Q (t=0)', zorder=5)
    ax_face.plot([px - x1, qx - x1], [py - y1, qy - y1], 'g-', linewidth=1.5, alpha=0.7)
    rx = t_actual * px + (1 - t_actual) * qx
    ry = t_actual * py + (1 - t_actual) * qy
    ax_face.plot(rx - x1, ry - y1, 'g*', markersize=12, label=f't={t_actual:.3f}', zorder=5)
    ax_face.set_title(
        f'idx={idx}  {part_name} {sub_key}\n'
        f'eye_scale={eye_scale:.1f}px  pq_ratio={pq_ratio:.3f}',
        fontsize=9,
    )
    ax_face.legend(loc='upper right', fontsize=7)
    ax_face.axis('off')

    # 右: パッチ群
    for i, (patch, t_aug) in enumerate(patches_to_show):
        ax = fig.add_subplot(gs[i // ncols_patch, 1 + (i % ncols_patch)])
        ax.imshow((patch * 255).clip(0, 255).astype(np.uint8), aspect='auto')
        for t_val, color, lbl, lw in [
            (1.0,   'blue', 'P',            1.5),
            (0.0,   'red',  'Q',            1.5),
            (t_aug, 'lime', f'{t_aug:.3f}', 2.0),
        ]:
            x_px = t_to_patch_x(t_val, patch_t_min, patch_t_max, patch_w)
            if 0 <= x_px <= patch_w:
                ax.axvline(x_px, color=color, linewidth=lw, linestyle='--', label=lbl)
        prefix = 'clean' if aug_flags == 0 else f'aug #{i + 1}'
        ax.set_title(f'{prefix}  t={t_aug:.3f}', fontsize=8)
        if i == 0:
            ax.legend(loc='upper right', fontsize=6)
        ax.axis('off')

    fig.text(
        0.5, 0.01,
        f'patch_t=[{patch_t_min:.1f},{patch_t_max:.1f}]  perp={patch_perp_scale:.1f}  '
        f'size={patch_h}x{patch_w}  label_t=[{label_t_min:.1f},{label_t_max:.1f}]  '
        f'aug_flags=0x{aug_flags:02X}({aug_flags})',
        ha='center', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
    )
    plt.show()


interactive_out = widgets.interactive_output(
    render,
    {
        'idx':              w_idx,
        'aug_flags':        w_aug_flags,
        'n_aug':            w_n_aug,
        'patch_t_min':      w_t_min,
        'patch_t_max':      w_t_max,
        'patch_perp_scale': w_perp,
        'patch_h':          w_h,
        'patch_w':          w_w,
        'label_t_min':      w_label_t_min,
        'label_t_max':      w_label_t_max,
    },
)

ui = widgets.VBox([
    widgets.HBox([w_idx, btn_rnd_all, btn_rnd_upper, btn_rnd_lower]),
    widgets.HBox([btn_rnd_edge, btn_rnd_small, btn_rnd_large]),
    widgets.HBox([w_aug_flags, w_n_aug]),
    widgets.HBox([w_t_min, w_t_max]),
    widgets.HBox([w_label_t_min, w_label_t_max]),
    widgets.HBox([w_perp]),
    widgets.HBox([w_h, w_w]),
])

display(ui, interactive_out)